In [3]:
# Core
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st

# ML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import shap

# Model saving
import joblib

/Users/DAP-User/Data_Science/DataAnalytics/Analytics/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
train_path = "./data/sandbox_capstone_train.csv"
test_path = "./data/sandbox_capstone_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df.head(15)

,Project Name,Project Year,Project Price,Project Timeline,Project Team Size,Project Team On-site work from home arrangement (%),Project Geographic Scope,Project Status,Updated Project Status
0,Beta Analytics,2019,527539.11,6 months,8,70.0,Region,Failed,Failed (high probability due to duration)
1,Alpha System 1,2018,503688.74,6 months,9,20.6,Region,Successful,Successful (but high risk)
2,Alpha Platform,2021,713488.51,6 months,6,17.7,City,Failed,Failed (high probability due to duration)
3,Alpha Integration,2023,293411.02,12 months,6,56.2,Institution,Failed,Failed (high probability due to duration)
4,Delta Expansion,2016,1717869.56,12 months,6,23.0,City,Failed,Failed (high probability due to duration)
5,Orion System,2024,1288207.66,10 months,7,31.6,Town,Failed,Failed (high probability due to duration)
6,Alpha Upgrade,2015,1619839.31,5 months,6,27.0,Nationwide,Failed,Failed (high probability due to duration)
7,Vertex Platform,2019,322279.12,12 months,7,81.8,Region,Successful,Successful (but high risk)
8,Delta Platform,2023,1012355.88,15 months,3,73.4,Region,Failed,Failed (high probability due to duration)
9,Apex Optimization,2024,1992436.69,20 months,10,64.4,Institution,Successful,Successful (but high risk)


In [21]:
def convert_timeline(value):
    try:
        # If it's already a number (int or float), return as int
        return int(value)
    except ValueError:
        # Otherwise, try to parse the first numeric part
        import re
        match = re.search(r'\d+', str(value))
        if match:
            return int(match.group())
        else:
            # If no number found, return NaN
            return np.nan

train_df["Timeline_Months"] = train_df["Project Timeline"].apply(convert_timeline)
test_df["Timeline_Months"] = test_df["Project Timeline"].apply(convert_timeline)

In [22]:
le = LabelEncoder()
train_df["Project Status"] = le.fit_transform(train_df["Project Status"])

In [23]:
features = [
    "Project Year",
    "Project Price",
    "Timeline_Months",
    "Project Team Size",
    "Project Team On-site work from home arrangement (%)",
    "Project Geographic Scope"
]

X = train_df[features]
y = train_df["Updated Project Status"]

In [24]:
region_mapping = {
    "Nationwide": 6,
    "City": 5,
    "Town": 4,
    "Barrangay": 3,
    "Institute": 2,
    "Office": 1
}

X["Project Geographic Scope"] = X["Project Geographic Scope"].map(region_mapping)

/var/folders/pk/xhnv_gdj5p70hwg8d6_f5l7m0000gp/T/ipykernel_3542/515888676.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Project Geographic Scope"] = X["Project Geographic Scope"].map(region_mapping)


In [25]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=50
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=50
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [26]:
pred = model.predict(X_val)
accuracy = accuracy_score(y_val, pred)

print("Accuracy:", accuracy)
print(classification_report(y_val, pred))

Accuracy: 1.0
                                           precision    recall  f1-score   support

                                   Failed       1.00      1.00      1.00         2
Failed (high probability due to duration)       1.00      1.00      1.00        18

                                 accuracy                           1.00        20
                                macro avg       1.00      1.00      1.00        20
                             weighted avg       1.00      1.00      1.00        20



In [27]:
explainer = shap.TreeExplainer(model)

In [ ]:
def evaluate_project(project):
    # ---- Encode Geographic Scope (must match training!) ----
    scope_mapping = {
        "Local": 1,
        "City": 2,
        "Regional": 3,
        "Nationwide": 4
    }

    encoded_scope = scope_mapping.get(project["Project Geographic Scope"], 4)

    # ---- Build DataFrame for prediction ----
    new_data = pd.DataFrame([{
        "Project Year": project["Project Year"],
        "Project Price": project["Project Price"],
        "Timeline_Months": project["Project Timeline"],
        "Project Team Size": project["Project Team Size"],
        "Project Team On-site work from home arrangement (%)": project["Onsite Percentage"],
        "Project Geographic Scope": encoded_scope
    }])

    # ---- Prediction ----
    probability = model.predict_proba(new_data)[0][1]
    probability_percent = round(probability * 100, 2)

    # ---- Risk Label ----
    if probability_percent >= 80:
        risk_label = "Low Risk"
    elif probability_percent >= 60:
        risk_label = "Moderate Risk"
    else:
        risk_label = "High Risk"

    # ---- Recommendations & Reasoning ----
    recommendations = []

    if project["Project Price"] < 5_000_000:
        recommendations.append("Funding is below optimal levels for project success.")

    if not (6 <= project["Project Timeline"] <= 8):
        recommendations.append("Project timeline is outside the optimal 6–8 month range.")

    if project["Project Team Size"] < 4:
        recommendations.append("Team size is too small for efficient execution. Have atleast 4 members in the group")

    if project["Project Team Size"] > 15:
        recommendations.append("Large team size may increase coordination complexity.")

    if project["Onsite Percentage"] < 70:
        recommendations.append("Low on-site collaboration may impact coordination.")

    if project["Project Geographic Scope"] not in ["Office", "Institution","Local", "City"]:
        recommendations.append("Wide geographic scope increases delivery risk.")

    if not recommendations:
        recommendations.append("Project configuration appears well-optimized.")

    # ---- Output ----
    print(f"\nProject Name: {project['Project Name']}")
    print(f"Success Probability: {probability_percent}%")
    print("\nReasoning & Recommendations:")
    print(f"Your Project Success Probability is at {probability_percent}% because of the following reasons:")

    for r in recommendations:
        print("-", r)

    print(f"*Risk Level: {risk_label}.*\n")
    print("NOTE: This is a simulated prediction and not a guaranteed outcome.\n")
    

    

In [69]:
project = {
    'Project Name': 'SimulAIt',
    'Project Year': 2026,
    'Project Price': 100000,
    'Project Timeline': 3, #in months
    'Project Team Size': 2, #in person
    'Onsite Percentage': 50, #Daily percentage of project staff working on site
    'Project Geographic Scope': 5 # 6 - Nationwide, 5 - City, 4 - Town, 3 - Barrangay, 2 - Institution, 1 - Office
}

evaluate_project(project)


Project Name: SimulAIt
Success Probability: 29.36%

Reasoning & Recommendations:
Your Project Success Probability is at 29.36% because of the following reasons:
- Funding is below optimal levels for project success.
- Project timeline is outside the optimal 6–8 month range.
- Team size is too small for efficient execution. Have atleast 4 members in the group
- Low on-site collaboration may impact coordination.
- Wide geographic scope increases delivery risk.
*Risk Level: High Risk.*

NOTE: This is a simulated prediction and not a guaranteed outcome.



AttributeError: 'list' object has no attribute 'to_df'

In [26]:
for name in dir(math):
    print(name, end="\t")
    

__doc__	__file__	__loader__	__name__	__package__	__spec__	acos	acosh	asin	asinh	atan	atan2	atanh	cbrt	ceil	comb	copysign	cos	cosh	degrees	dist	e	erf	erfc	exp	exp2	expm1	fabs	factorial	floor	fma	fmod	frexp	fsum	gamma	gcd	hypot	inf	isclose	isfinite	isinf	isnan	isqrt	lcm	ldexp	lgamma	log	log10	log1p	log2	modf	nan	nextafter	perm	pi	pow	prod	radians	remainder	sin	sinh	sqrt	sumprod	tan	tanh	tau	trunc	ulp	

In [65]:
from math import e, log, exp
show = dir(math)
print(show, end="\n")

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt', 'ceil', 'comb', 'copysign', 'cos', 'cosh', 'degrees', 'dist', 'e', 'erf', 'erfc', 'exp', 'exp2', 'expm1', 'fabs', 'factorial', 'floor', 'fma', 'fmod', 'frexp', 'fsum', 'gamma', 'gcd', 'hypot', 'inf', 'isclose', 'isfinite', 'isinf', 'isnan', 'isqrt', 'lcm', 'ldexp', 'lgamma', 'log', 'log10', 'log1p', 'log2', 'modf', 'nan', 'nextafter', 'perm', 'pi', 'pow', 'prod', 'radians', 'remainder', 'sin', 'sinh', 'sqrt', 'sumprod', 'tan', 'tanh', 'tau', 'trunc', 'ulp']


In [73]:
math.exp(2 * log(2))

4.0

In [74]:
exp(2 * log(2))

4.0